# Exploratory data analysis

Exploration is how a dataset tells you what questions it can answer. This notebook looks at the
distribution of the target, how features correlate with it and with each other, and what the
geography adds. The aim is understanding, not decoration.

## Learning objectives

By the end of this notebook you will be able to:

- summarise a distribution with quantiles and a histogram;
- rank features by correlation with the target and explain what correlation does not mean;
- spot multicollinearity between features;
- bin a numeric feature and compare group means;
- read a map-like scatter of latitude and longitude.

## Concept

A **distribution** tells you the shape of a variable: where it is centred, how spread out it is,
and whether it is skewed. The mean alone hides all of that, so look at quantiles and a histogram
together. A long right tail means a few large values dominate the mean.

**Correlation** measures how strongly two variables move together on a straight line, from -1 to
+1. It is not causation, and it only captures linear relationships; two variables can be strongly
related in a curve with a correlation near zero. **Multicollinearity** is when features carry
nearly the same information, which makes linear model coefficients unstable and hard to interpret
even though predictions may be fine.

**Binning** a continuous feature (for example income into quartiles) turns a noisy relationship
into readable group means. It is a presentation choice, not a model: information is lost at the
boundaries.

Because the data has latitude and longitude, plotting them reveals spatial clusters that no
single summary statistic can show.

## Worked example

### Load and set the theme

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
import analysis
from ds_practice import load_california, set_seed, set_theme, histogram, scatterplot, lineplot

set_seed(42)
set_theme()
housing = load_california()
print("shape:", housing.shape)

### The target distribution

The histogram shows the cap at 5.0 as a spike on the right, and the quantiles show the bulk of
block groups sit between roughly 1.2 and 2.6 hundred-thousand dollars.

In [ ]:
print(housing["MedHouseVal"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(3).to_string())
fig, ax = histogram(housing["MedHouseVal"], bins=40,
                    title="Median house value", xlabel="MedHouseVal (100k USD)")

### Correlation with the target

A correlation table ranks which single features move most closely with the target. Median income
is usually the strongest, with geography close behind.

In [ ]:
corr_with_target = (
    housing.corr(numeric_only=True)["MedHouseVal"]
    .drop("MedHouseVal")
    .sort_values(key=abs, ascending=False)
)
print(corr_with_target.round(3).to_string())

### Correlated features

Some predictors are strongly correlated with each other. `AveRooms` and `AveBedrms` move together,
as do `Latitude`/`Longitude` with the target. This is the multicollinearity to watch.

In [ ]:
feature_corr = housing[analysis.BASE_FEATURES].corr().round(2)
display(feature_corr.loc[["AveRooms", "AveBedrms", "Population", "AveOccup"],
                         ["AveRooms", "AveBedrms", "Population", "AveOccup"]])

### Binning income

Quartiles of median income turn a noisy scatter into a clean relationship: average house value
rises steadily with income.

In [ ]:
housing = housing.assign(income_quartile=pd.qcut(housing["MedInc"], 4, labels=["Q1", "Q2", "Q3", "Q4"]))
by_income = housing.groupby("income_quartile", observed=True)["MedHouseVal"].agg(["count", "mean"])
display(by_income.round(3))
fig, ax = lineplot(by_income.index.astype(str), by_income["mean"],
                   title="Mean house value by income quartile",
                   xlabel="Income quartile", ylabel="Mean MedHouseVal")

### Geography

Plotting longitude against latitude, coloured by value, shows coastal clusters where values are
highest. `scatterplot` keeps the example dependency-free.

In [ ]:
fig, ax = scatterplot(housing["Longitude"], housing["Latitude"],
                      title="Block-group geography (colour = value)",
                      xlabel="Longitude", ylabel="Latitude")
fig, ax = scatterplot(housing["MedInc"], housing["MedHouseVal"],
                      title="Income vs house value", xlabel="MedInc", ylabel="MedHouseVal")

## Exercises

1. **Skew check.** Compare the mean and median of `Population` and `AveOccup`. Which is more
   skewed, and what does that imply for a model that assumes symmetry?
2. **Non-linear pair.** Find a feature whose correlation with the target is weak but whose
   binned means are clearly ordered. Explain why correlation missed it.
3. **Coastal effect.** Create a flag for `Latitude < 37` and compare mean `MedHouseVal` on each
   side. Report the difference and one reason the split is crude.

## Limitations

Correlations here are computed over block groups, which aggregate very different households, so
they can differ from individual-level relationships. The capped target biases every downward-facing
summary. Visual inspection of a two-dimensional scatter cannot capture more than two or three
variables at once, and the "coastal" pattern is a proxy for many unmeasured factors rather than a
cause. EDA guides modelling choices; it does not prove them.